# Spatial Autocorrelation: Global Moran's I and LISA

This notebook analyzes spatial autocorrelation in Indonesian provincial poverty rates using Global Moran's I and Local Indicators of Spatial Association (LISA). The analysis uses BPS panel data for 2021-2024 and centroid-based spatial weights.

In [23]:
import plotly.express as px
import plotly.graph_objects as go

## Library Imports

In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import seaborn as sns
from scipy import stats

import libpysal
from libpysal.weights import KNN
from esda.moran import Moran, Moran_Local

from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / 'data_bps_datmin.csv'
OUTPUT_DIR = Path('spatial_output')
OUTPUT_DIR.mkdir(exist_ok=True)

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

print('All libraries were imported successfully.')
print(f'libpysal version: {libpysal.__version__}')


All libraries were imported successfully.
libpysal version: 4.14.1


## Data Loading and Exploration

In [25]:
df_raw = pd.read_csv(DATA_PATH)
df_raw.columns = [
    'province', 'year', 'aps_1315', 'aps_1618', 'aps_1924',
    'tpt_feb', 'tpt_aug', 'tpak_feb', 'tpak_aug',
    'poverty_line_march', 'poverty_line_september',
    'poor_population_march', 'poor_population_september',
    'poverty_pct_march', 'poverty_pct_september',
    'hdi', 'mean_years_schooling', 'expected_years_schooling'
]

print(f'Rows: {df_raw.shape[0]}')
print(f'Columns: {df_raw.shape[1]}')
print(f'Unique provinces: {df_raw["province"].nunique()}')
print(f'Years: {sorted(df_raw["year"].unique())}')
print()
print(df_raw.columns.tolist())
print()
df_raw.head()


Rows: 151
Columns: 18
Unique provinces: 38
Years: [2021, 2022, 2023, 2024]

['province', 'year', 'aps_1315', 'aps_1618', 'aps_1924', 'tpt_feb', 'tpt_aug', 'tpak_feb', 'tpak_aug', 'poverty_line_march', 'poverty_line_september', 'poor_population_march', 'poor_population_september', 'poverty_pct_march', 'poverty_pct_september', 'hdi', 'mean_years_schooling', 'expected_years_schooling']



,province,year,aps_1315,aps_1618,aps_1924,tpt_feb,tpt_aug,tpak_feb,tpak_aug,poverty_line_march,poverty_line_september,poor_population_march,poor_population_september,poverty_pct_march,poverty_pct_september,hdi,mean_years_schooling,expected_years_schooling
0,SUMATERA UTARA,2021,96.99,78.66,27.05,6.01,6.33,69.39,69.10,525756,648336,1343.86,1110.92,9.01,7.19,73.84,9.88,13.27
1,SUMATERA BARAT,2021,96.63,84.07,36.41,6.67,6.52,68.41,67.72,568703,714991,370.67,315.43,6.63,5.42,74.56,9.46,14.09
2,RIAU,2021,95.66,77.81,28.79,4.96,4.42,65.81,65.03,565937,702620,500.81,473.04,7.12,6.36,73.89,9.52,13.28
3,JAMBI,2021,96.39,72.50,24.14,4.76,5.09,67.30,67.17,506355,658100,293.86,272.70,8.09,7.26,72.62,9.03,13.04
4,SUMATERA SELATAN,2021,94.85,71.53,18.81,5.17,4.98,69.95,68.77,457455,564462,1113.76,948.84,12.84,10.51,71.83,8.78,12.54


In [26]:
desc = df_raw.describe()
print(desc.to_string())


              year    aps_1315    aps_1618    aps_1924     tpt_feb     tpt_aug    tpak_feb    tpak_aug  poverty_line_march  poverty_line_september  poor_population_march  poor_population_september  poverty_pct_march  poverty_pct_september         hdi  mean_years_schooling  expected_years_schooling
count   151.000000  151.000000  151.000000  151.000000  151.000000  151.000000  151.000000  151.000000        1.510000e+02              151.000000             151.000000                 151.000000         151.000000             151.000000  151.000000            151.000000                151.000000
mean   2022.509934   94.392318   74.570331   27.616159    4.952848    4.764834   69.486623   69.577219        5.838603e+05           652458.443709             747.771126                 626.132715          11.704901              10.651656   72.264106              9.119536                 13.133046
std       1.118735    5.109407    6.389758    6.132170    1.523588    1.614804    4.031956    4.124931 

In [27]:
print('=== Missing Values by Column ===')
missing = df_raw.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values.')
print()

dup = df_raw.duplicated(subset=['province', 'year']).sum()
print(f'Duplicates (Province x Year): {dup}')


=== Missing Values by Column ===
No missing values.

Duplicates (Province x Year): 0


## Data Preprocessing

In [28]:
df = df_raw.copy()
df.columns = [
    'province', 'year', 'aps_1315', 'aps_1618', 'aps_1924',
    'tpt_feb', 'tpt_aug', 'tpak_feb', 'tpak_aug',
    'poverty_line_march', 'poverty_line_september',
    'poor_population_march', 'poor_population_september',
    'poverty_pct_march', 'poverty_pct_september',
    'hdi', 'mean_years_schooling', 'expected_years_schooling'
]

df['poverty_rate'] = (df['poverty_pct_march'] + df['poverty_pct_september']) / 2

df['tpt_avg'] = (df['tpt_feb'] + df['tpt_aug']) / 2

df['tpak_avg'] = (df['tpak_feb'] + df['tpak_aug']) / 2

df['province'] = df['province'].str.strip().str.upper()

print('=== Dataset after preprocessing ===')
print(f'Shape: {df.shape}')
print(df[['province', 'year', 'poverty_rate', 'hdi', 'tpt_avg']].head(10).to_string())


=== Dataset after preprocessing ===
Shape: (151, 21)
               province  year  poverty_rate    hdi  tpt_avg
0        SUMATERA UTARA  2021         8.100  73.84    6.170
1        SUMATERA BARAT  2021         6.025  74.56    6.595
2                  RIAU  2021         6.740  73.89    4.690
3                 JAMBI  2021         7.675  72.62    4.925
4      SUMATERA SELATAN  2021        11.675  71.83    5.075
5              BENGKULU  2021        13.870  73.16    3.685
6               LAMPUNG  2021        11.620  71.25    4.615
7  KEP. BANGKA BELITUNG  2021         4.990  72.96    5.035
8             KEP. RIAU  2021         5.450  77.87   10.015
9           DKI JAKARTA  2021         4.430  82.25    8.505


In [29]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

years = sorted(df['year'].unique())
prov_avg = df.groupby('province')['poverty_rate'].mean().sort_values(ascending=True)
top10 = prov_avg.tail(10)

fig_dist = make_subplots(rows=1, cols=2, subplot_titles=('Poverty Rate Distribution by Year', 'Top 10 Provinces by Avg Poverty Rate (2021-2024)'))

fig_dist.add_trace(go.Box(x=df['year'], y=df['poverty_rate'], marker_color='#4472C4', name='Poverty Rate'), row=1, col=1)

colors_bar = ['#C00000' if v >= prov_avg.median() * 1.5 else '#4472C4' for v in top10.values]
fig_dist.add_trace(go.Bar(y=top10.index, x=top10.values, orientation='h', marker_color=colors_bar, name='Avg Poverty Rate'), row=1, col=2)

fig_dist.update_layout(height=500, width=1100, showlegend=False, title_text="Poverty Distribution")
fig_dist.write_json(OUTPUT_DIR / 'plot_poverty_distribution.json')
fig_dist.show()

print('\nTop 10 Provinces by Average Poverty Rate (2021-2024)')
print(top10.sort_values(ascending=False).round(2).to_string())



Top 10 Provinces by Average Poverty Rate (2021-2024)
province
PAPUA PEGUNUNGAN       28.88
PAPUA TENGAH           27.45
PAPUA SELATAN          21.79
PAPUA BARAT            21.21
PAPUA                  21.13
NUSA TENGGARA TIMUR    19.57
PAPUA BARAT DAYA       18.70
MALUKU                 16.18
GORONTALO              14.53
ACEH                   13.54


## Spatial Weight Matrix Construction

Because no shapefile is included in this project folder, the spatial structure is approximated using manually defined provincial centroid coordinates. The main spatial weight matrix uses K-Nearest Neighbors with `k=5`.

In [30]:
province_COORDS = {
    'ACEH': (95.317, 4.695),
    'SUMATERA UTARA': (98.672, 2.115),
    'SUMATERA BARAT': (100.355, -0.744),
    'RIAU': (101.447, 0.293),
    'JAMBI': (102.437, -1.609),
    'SUMATERA SELATAN': (104.761, -3.319),
    'BENGKULU': (102.346, -3.800),
    'LAMPUNG': (105.322, -4.558),
    'KEPULAUAN BANGKA BELITUNG': (106.116, -2.741),
    'KEPULAUAN RIAU': (104.030, 0.921),
    'DKI JAKARTA': (106.845, -6.208),
    'JAWA BARAT': (107.619, -6.902),
    'JAWA TENGAH': (110.165, -7.150),
    'DI YOGYAKARTA': (110.365, -7.800),
    'JAWA TIMUR': (112.752, -7.536),
    'BANTEN': (106.064, -6.406),
    'BALI': (115.188, -8.409),
    'NUSA TENGGARA BARAT': (116.419, -8.600),
    'NUSA TENGGARA TIMUR': (121.079, -8.657),
    'KALIMANTAN BARAT': (109.697, -0.023),
    'KALIMANTAN TENGAH': (113.941, -1.681),
    'KALIMANTAN SELATAN': (115.283, -3.093),
    'KALIMANTAN TIMUR': (116.419, 1.681),
    'KALIMANTAN UTARA': (116.593, 3.073),
    'SULAWESI UTARA': (124.841, 0.632),
    'SULAWESI TENGAH': (121.445, -1.431),
    'SULAWESI SELATAN': (120.190, -3.668),
    'SULAWESI TENGGARA': (122.390, -4.145),
    'GORONTALO': (122.446, 0.541),
    'SULAWESI BARAT': (119.313, -2.840),
    'MALUKU': (128.175, -3.238),
    'MALUKU UTARA': (127.624, 1.571),
    'PAPUA BARAT': (133.173, -1.336),
    'PAPUA': (138.379, -4.269),
    'PAPUA PEGUNUNGAN': (138.600, -4.000),
    'PAPUA SELATAN': (138.500, -6.500),
    'PAPUA TENGAH': (136.500, -3.500),
    'PAPUA BARAT DAYA': (131.500, -1.500)
}

coords_df = pd.DataFrame(
    [(k, v[0], v[1]) for k, v in province_COORDS.items()],
    columns=['province', 'lon', 'lat']
)

print(f'Total provinces with coordinates: {len(coords_df)}')

prov_in_data = set(df['province'].unique())
prov_in_coords = set(coords_df['province'].unique())
missing_coords = prov_in_data - prov_in_coords
print(f'\nProvinces in data but missing coordinates: {missing_coords}')

missing_data = prov_in_coords - prov_in_data
print(f'Provinces in coordinates but missing from data: {missing_data}')


Total provinces with coordinates: 38

Provinces in data but missing coordinates: {'KEP. BANGKA BELITUNG', 'KEP. RIAU'}
Provinces in coordinates but missing from data: {'KEPULAUAN RIAU', 'KEPULAUAN BANGKA BELITUNG'}


In [31]:
PROV_MAP = {
    'KEPULAUAN BANGKA BELITUNG': 'KEP. BANGKA BELITUNG',
    'KEPULAUAN RIAU': 'KEP. RIAU',
    'DKI JAKARTA': 'DKI JAKARTA',
    'DI YOGYAKARTA': 'DI YOGYAKARTA',
    'NUSA TENGGARA BARAT': 'NUSA TENGGARA BARAT',
    'NUSA TENGGARA TIMUR': 'NUSA TENGGARA TIMUR',
}

print('Province names in the dataset:')
for p in sorted(df['province'].unique()):
    matched = p in province_COORDS
    print(f'  {p:45s} -> {"OK" if matched else "NO MATCH"}')


Province names in the dataset:
  ACEH                                          -> OK
  BALI                                          -> OK
  BANTEN                                        -> OK
  BENGKULU                                      -> OK
  DI YOGYAKARTA                                 -> OK
  DKI JAKARTA                                   -> OK
  GORONTALO                                     -> OK
  JAMBI                                         -> OK
  JAWA BARAT                                    -> OK
  JAWA TENGAH                                   -> OK
  JAWA TIMUR                                    -> OK
  KALIMANTAN BARAT                              -> OK
  KALIMANTAN SELATAN                            -> OK
  KALIMANTAN TENGAH                             -> OK
  KALIMANTAN TIMUR                              -> OK
  KALIMANTAN UTARA                              -> OK
  KEP. BANGKA BELITUNG                          -> NO MATCH
  KEP. RIAU                                  

In [32]:
def get_coords(prov_name, coords_dict):
    if prov_name in coords_dict:
        return coords_dict[prov_name]
    for key in coords_dict:
        if prov_name in key or key in prov_name:
            return coords_dict[key]
    return None

df['lon'] = df['province'].apply(lambda x: get_coords(x, {k: v[0] for k, v in province_COORDS.items()}))
df['lat'] = df['province'].apply(lambda x: get_coords(x, {k: v[1] for k, v in province_COORDS.items()}))

no_coords = df[df['lon'].isnull()]['province'].unique()
print(f'Provinces without coordinates: {no_coords}')

df = df.dropna(subset=['lon', 'lat'])
print(f'\nShape after coordinate filtering: {df.shape}')


Provinces without coordinates: ['KEP. BANGKA BELITUNG']

Shape after coordinate filtering: (147, 23)


In [33]:
df_2023 = df[df['year'] == 2023].copy().reset_index(drop=True)

print(f'2023 data: {df_2023.shape[0]} provinces')
print(df_2023[['province', 'poverty_rate', 'lon', 'lat']].to_string())


2023 data: 37 provinces
               province  poverty_rate      lon    lat
0                  ACEH        13.545   95.317  4.695
1        SUMATERA UTARA         7.670   98.672  2.115
2        SUMATERA BARAT         5.685  100.355 -0.744
3                  RIAU         6.520  101.447  0.293
4                 JAMBI         7.420  102.437 -1.609
5      SUMATERA SELATAN        11.145  104.761 -3.319
6              BENGKULU        13.280  102.346 -3.800
7               LAMPUNG        10.865  105.322 -4.558
8             KEP. RIAU         5.235  101.447  0.293
9           DKI JAKARTA         4.290  106.845 -6.208
10           JAWA BARAT         7.350  107.619 -6.902
11          JAWA TENGAH        10.175  110.165 -7.150
12        DI YOGYAKARTA        10.720  110.365 -7.800
13           JAWA TIMUR         9.955  112.752 -7.536
14               BANTEN         5.935  106.064 -6.406
15                 BALI         4.025  115.188 -8.409
16  NUSA TENGGARA BARAT        12.880  116.419 -8.600
17  

In [34]:
coords_arr = list(zip(df_2023['lon'].values, df_2023['lat'].values))

W_knn = KNN.from_array(coords_arr, k=5)
W_knn.transform = 'r'

print('=== KNN Spatial Weights Matrix (k=5) ===')
print(f'Number of observations: {W_knn.n}')
print(f'Average neighbors: {W_knn.mean_neighbors:.2f}')
print(f'Minimum neighbors: {W_knn.min_neighbors}')
print(f'Maximum neighbors: {W_knn.max_neighbors}')
print(f'Sparsity: {W_knn.pct_nonzero:.2f}%')


=== KNN Spatial Weights Matrix (k=5) ===
Number of observations: 37
Average neighbors: 5.00
Minimum neighbors: 5
Maximum neighbors: 5
Sparsity: 13.51%


In [35]:
fig_map = go.Figure()

lons = df_2023['lon'].values
lats = df_2023['lat'].values
pov_vals = df_2023['poverty_rate'].values

# Add lines
for i, neighbors in W_knn.neighbors.items():
    for j in neighbors:
        fig_map.add_trace(go.Scatter(x=[lons[i], lons[j]], y=[lats[i], lats[j]], mode='lines', line=dict(color='gray', width=0.7), opacity=0.3, showlegend=False))

# Add nodes
fig_map.add_trace(go.Scatter(x=lons, y=lats, mode='markers+text', marker=dict(size=12, color=pov_vals, colorscale='RdYlGn_r', showscale=True, colorbar=dict(title='Poverty Rate (%)')), text=[p.replace(' ', '<br>') for p in df_2023['province']], textposition='top center', textfont=dict(size=8), showlegend=False))

fig_map.update_layout(title='Indonesian Provincial Spatial Connectivity Map (KNN k=5, 2023 Poverty Rate)', height=700, width=1200, xaxis_title='Longitude', yaxis_title='Latitude')
fig_map.write_json(OUTPUT_DIR / 'plot_spatial_connectivity.json')
fig_map.show()


## Global Moran's I

Global Moran's I measures whether poverty rates are spatially clustered across provinces. Positive values indicate neighboring provinces tend to have similar poverty levels.

In [36]:
results_global = []

for year in sorted(df['year'].unique()):
    df_t = df[df['year'] == year].copy().reset_index(drop=True)
    coords_t = list(zip(df_t['lon'].values, df_t['lat'].values))
    W_t = KNN.from_array(coords_t, k=5)
    W_t.transform = 'r'

    y = df_t['poverty_rate'].values

    moran = Moran(y, W_t, permutations=999)

    results_global.append({
        'year': year,
        'n_province': len(df_t),
        'moran_i': moran.I,
        'expected_i': moran.EI,
        'z_score': moran.z_norm,
        'p_value': moran.p_norm,
        'p_sim': moran.p_sim,
        'significant': moran.p_sim < 0.05
    })

    print(f'\nYear {year}:')
    print(f'  Moran\'s I      = {moran.I:.4f}')
    print(f'  Expected I    = {moran.EI:.4f}')
    print(f'  Z-score       = {moran.z_norm:.4f}')
    print(f'  p-value (norm)= {moran.p_norm:.4f}')
    print(f'  p-value (sim) = {moran.p_sim:.4f}')
    print(f'  Significant?   = {moran.p_sim < 0.05}')
    if moran.I > 0 and moran.p_sim < 0.05:
        print(f'  Interpretation  : There is significant positive spatial clustering')
    elif moran.I < 0 and moran.p_sim < 0.05:
        print(f'  Interpretation  : There is significant spatial dispersion')
    else:
        print(f'  Interpretation  : Pola ACAK (not significant)')

df_global = pd.DataFrame(results_global)
print("\n=== Global Moran's I Summary ===")
print(df_global.to_string(index=False))



Year 2021:
  Moran's I      = 0.6322
  Expected I    = -0.0286
  Z-score       = 7.1944
  p-value (norm)= 0.0000
  p-value (sim) = 0.0010
  Significant?   = True
  Interpretation  : There is significant positive spatial clustering

Year 2022:
  Moran's I      = 0.6331
  Expected I    = -0.0278
  Z-score       = 7.3056
  p-value (norm)= 0.0000
  p-value (sim) = 0.0010
  Significant?   = True
  Interpretation  : There is significant positive spatial clustering

Year 2023:
  Moran's I      = 0.6303
  Expected I    = -0.0278
  Z-score       = 7.2750
  p-value (norm)= 0.0000
  p-value (sim) = 0.0010
  Significant?   = True
  Interpretation  : There is significant positive spatial clustering

Year 2024:
  Moran's I      = 0.5685
  Expected I    = -0.0278
  Z-score       = 6.5922
  p-value (norm)= 0.0000
  p-value (sim) = 0.0010
  Significant?   = True
  Interpretation  : There is significant positive spatial clustering

=== Global Moran's I Summary ===
 year  n_province  moran_i  expected_i

In [37]:
years = sorted(df['year'].unique())
fig_scatter = make_subplots(rows=2, cols=2, subplot_titles=[f'Moran Scatter Plot {y}' for y in years])

for idx, year in enumerate(years):
    row_idx = (idx // 2) + 1
    col_idx = (idx % 2) + 1
    
    df_t = df[df['year'] == year].copy().reset_index(drop=True)
    coords_t = list(zip(df_t['lon'].values, df_t['lat'].values))
    W_t = KNN.from_array(coords_t, k=5)
    W_t.transform = 'r'

    y = df_t['poverty_rate'].values
    moran = Moran(y, W_t, permutations=999)

    y_std = (y - y.mean()) / y.std()
    lag_y_std = libpysal.weights.lag_spatial(W_t, y_std)

    colors = []
    for yi, lyi in zip(y_std, lag_y_std):
        if yi >= 0 and lyi >= 0: colors.append('#C00000') # HH
        elif yi < 0 and lyi < 0: colors.append('#4472C4') # LL
        elif yi >= 0 and lyi < 0: colors.append('#FF7F00') # HL
        else: colors.append('#70AD47') # LH

    fig_scatter.add_trace(go.Scatter(x=y_std, y=lag_y_std, mode='markers', marker=dict(color=colors, size=8, line=dict(width=1, color='black')), showlegend=False), row=row_idx, col=col_idx)
    
    # regression line
    slope, intercept, r, p, se = stats.linregress(y_std, lag_y_std)
    x_line = np.array([y_std.min(), y_std.max()])
    fig_scatter.add_trace(go.Scatter(x=x_line, y=slope*x_line + intercept, mode='lines', line=dict(color='black', width=1.5), showlegend=False), row=row_idx, col=col_idx)

fig_scatter.update_layout(height=800, width=1000, title_text='Indonesian Provincial Poverty Moran Scatter Plot 2021-2024')
fig_scatter.write_json(OUTPUT_DIR / 'plot_moran_scatter.json')
fig_scatter.show()


In [38]:
fig_trend = make_subplots(rows=1, cols=2, subplot_titles=("Trend of Global Moran's I (2021-2024)", "Z-score Moran's I (2021-2024)"))

# Subplot 1
fig_trend.add_trace(go.Scatter(x=df_global['year'], y=df_global['moran_i'], mode='lines+markers', marker=dict(size=10, color='blue'), name="Moran's I"), row=1, col=1)
fig_trend.add_hline(y=df_global['expected_i'].mean(), line_dash="dash", line_color="gray", annotation_text="Expected I", row=1, col=1)

# Subplot 2
colors_z = ['#C00000' if s else '#4472C4' for s in df_global['significant']]
fig_trend.add_trace(go.Bar(x=df_global['year'], y=df_global['z_score'], marker_color=colors_z, name="Z-score"), row=1, col=2)
fig_trend.add_hline(y=1.96, line_dash="dash", line_color="red", annotation_text="Z kritis = 1.96", row=1, col=2)
fig_trend.add_hline(y=-1.96, line_dash="dash", line_color="red", row=1, col=2)

fig_trend.update_layout(height=500, width=1100, title_text='Global Moran Analysis')
fig_trend.write_json(OUTPUT_DIR / 'plot_moran_trend.json')
fig_trend.show()


## Local Indicators of Spatial Association (LISA)

LISA identifies local poverty clusters and spatial outliers for each province. The categories are High-High, Low-Low, High-Low, Low-High, and non-significant.

In [39]:
lisa_results_all = []

LISA_COLORS = {
    'HH': '#C00000',
    'LL': '#4472C4',
    'HL': '#FF7F00',
    'LH': '#70AD47',
    'NS': '#D3D3D3'
}

for year in sorted(df['year'].unique()):
    df_t = df[df['year'] == year].copy().reset_index(drop=True)
    coords_t = list(zip(df_t['lon'].values, df_t['lat'].values))
    W_t = KNN.from_array(coords_t, k=5)
    W_t.transform = 'r'

    y = df_t['poverty_rate'].values

    lisa = Moran_Local(y, W_t, permutations=999)

    y_std = (y - y.mean()) / y.std()
    lag_y_std = libpysal.weights.lag_spatial(W_t, y_std)

    sig_mask = lisa.p_sim < 0.05
    quadrant = []
    for i in range(len(y_std)):
        if not sig_mask[i]:
            quadrant.append('NS')
        elif y_std[i] >= 0 and lag_y_std[i] >= 0:
            quadrant.append('HH')
        elif y_std[i] < 0 and lag_y_std[i] < 0:
            quadrant.append('LL')
        elif y_std[i] >= 0 and lag_y_std[i] < 0:
            quadrant.append('HL')
        else:
            quadrant.append('LH')

    for i, row in df_t.iterrows():
        lisa_results_all.append({
            'year': year,
            'province': row['province'],
            'poverty_rate': row['poverty_rate'],
            'local_moran_i': lisa.Is[i],
            'p_value': lisa.p_sim[i],
            'quadrant': quadrant[i],
            'significant': sig_mask[i],
            'lon': row['lon'],
            'lat': row['lat']
        })

df_lisa = pd.DataFrame(lisa_results_all)

print('Hasil LISA by Year')
for year in sorted(df_lisa['year'].unique()):
    sub = df_lisa[df_lisa['year'] == year]
    print(f'\nYear {year}:')
    count_sig = sub[sub['significant']].groupby('quadrant').size()
    count_all = sub.groupby('quadrant').size()
    print(f'  Cluster significant (p<0.05): {sub["significant"].sum()} dari {len(sub)} province')
    print('  Distribution quadrant (significant):')
    for k in ['HH', 'LL', 'HL', 'LH', 'NS']:
        n = count_all.get(k, 0)
        if n > 0:
            print(f'    {k}: {n} province')
    sig_df = sub[sub['significant']].sort_values('quadrant')
    if len(sig_df) > 0:
        print('  Province significant:')
        for _, r in sig_df.iterrows():
            print(f'    [{r["quadrant"]}] {r["province"]} (I={r["local_moran_i"]:.4f}, p={r["p_value"]:.4f})')


Hasil LISA by Year

Year 2021:
  Cluster significant (p<0.05): 6 dari 36 province
  Distribution quadrant (significant):
    HH: 5 province
    LL: 1 province
    NS: 30 province
  Province significant:
    [HH] PAPUA BARAT (I=2.4988, p=0.0010)
    [HH] PAPUA (I=3.0134, p=0.0010)
    [HH] PAPUA SELATAN (I=3.1553, p=0.0010)
    [HH] PAPUA TENGAH (I=3.9964, p=0.0010)
    [HH] PAPUA PEGUNUNGAN (I=4.1824, p=0.0010)
    [LL] KALIMANTAN TENGAH (I=0.6788, p=0.0330)

Year 2022:
  Cluster significant (p<0.05): 8 dari 37 province
  Distribution quadrant (significant):
    HH: 5 province
    LL: 2 province
    HL: 1 province
    NS: 29 province
  Province significant:
    [HH] PAPUA BARAT (I=2.5372, p=0.0010)
    [HH] PAPUA (I=3.1303, p=0.0010)
    [HH] PAPUA SELATAN (I=3.2771, p=0.0010)
    [HH] PAPUA TENGAH (I=4.1470, p=0.0010)
    [HH] PAPUA PEGUNUNGAN (I=4.3393, p=0.0010)
    [HL] ACEH (I=-0.2502, p=0.0210)
    [LL] KALIMANTAN BARAT (I=0.4805, p=0.0450)
    [LL] KALIMANTAN TENGAH (I=0.6650, p

In [40]:
years = sorted(df_lisa['year'].unique())
fig_lisa = make_subplots(rows=2, cols=2, subplot_titles=[f'LISA Poverty Map {y}' for y in years])

LISA_COLORS = {'HH': '#C00000', 'LL': '#4472C4', 'HL': '#FF7F00', 'LH': '#70AD47', 'NS': '#D3D3D3'}

for idx, year in enumerate(years):
    row_idx = (idx // 2) + 1
    col_idx = (idx % 2) + 1
    sub = df_lisa[df_lisa['year'] == year]

    colors = [LISA_COLORS[q] for q in sub['quadrant']]
    sizes = [15 if sig else 7 for sig in sub['significant']]
    
    fig_lisa.add_trace(go.Scatter(x=sub['lon'], y=sub['lat'], mode='markers+text', text=[p.split()[0] if sig else '' for p, sig in zip(sub['province'], sub['significant'])], textposition='top center', marker=dict(color=colors, size=sizes, line=dict(width=1, color='black')), showlegend=False), row=row_idx, col=col_idx)

fig_lisa.update_layout(height=800, width=1100, title_text='LISA Poverty Map Province Indonesia 2021-2024')
fig_lisa.write_json(OUTPUT_DIR / 'plot_lisa_map.json')
fig_lisa.show()


In [41]:
import matplotlib.colors as mcolors

pivot_quadrant = df_lisa.pivot(index='province', columns='year', values='quadrant')

QUADRANT_CODE = {'HH': 4, 'LH': 3, 'HL': 2, 'LL': 1, 'NS': 0}
pivot_num = pivot_quadrant.applymap(lambda x: QUADRANT_CODE.get(x, 0))

ever_sig = df_lisa[df_lisa['significant']]['province'].unique()
pivot_num_sig = pivot_num.loc[pivot_num.index.isin(ever_sig)]

if len(pivot_num_sig) > 0:
    fig, ax = plt.subplots(figsize=(10, max(6, len(pivot_num_sig) * 0.4)))

    cmap = mcolors.ListedColormap(['#D3D3D3', '#4472C4', '#FF7F00', '#70AD47', '#C00000'])
    bounds = [-0.5, 0.5, 1.5, 2.5, 3.5, 4.5]
    norm = mcolors.BoundaryNorm(bounds, cmap.N)

    im = ax.imshow(pivot_num_sig.values, cmap=cmap, norm=norm, aspect='auto')

    ax.set_xticks(range(len(pivot_num_sig.columns)))
    ax.set_xticklabels(pivot_num_sig.columns)
    ax.set_yticks(range(len(pivot_num_sig.index)))
    ax.set_yticklabels(pivot_num_sig.index, fontsize=8)

    for i in range(len(pivot_num_sig.index)):
        for j in range(len(pivot_num_sig.columns)):
            val = pivot_num_sig.values[i, j]
            label = {v: k for k, v in QUADRANT_CODE.items()}[val]
            ax.text(j, i, label, ha='center', va='center', fontsize=9, fontweight='bold',
                    color='white' if val in [1, 4] else 'black')

    cbar = plt.colorbar(im, ax=ax, ticks=[0, 1, 2, 3, 4])
    cbar.ax.set_yticklabels(['NS', 'LL', 'HL', 'LH', 'HH'])

    ax.set_title('LISA Quadrant Consistency per Province (2021-2024)\n(only provinces significant at least once)')
    ax.set_xlabel('Year')
    ax.set_ylabel('Province')

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'plot_lisa_consistency.png', bbox_inches='tight')
    plt.show()
else:
    print('There is no province that was ever significant in LISA.')

In [42]:
fig_dist_lisa = make_subplots(rows=1, cols=2, subplot_titles=("Distribution Local Moran's I by Year", "Number of Provinces per Tipe Cluster LISA"))

fig_dist_lisa.add_trace(go.Box(x=df_lisa['year'], y=df_lisa['local_moran_i'], marker_color='#4472C4', name='Local Moran I'), row=1, col=1)

count_cluster = df_lisa[df_lisa['significant']].groupby(['year', 'quadrant']).size().unstack(fill_value=0)
for col in count_cluster.columns:
    fig_dist_lisa.add_trace(go.Bar(name=col, x=count_cluster.index, y=count_cluster[col], marker_color=LISA_COLORS.get(col, 'gray')), row=1, col=2)

fig_dist_lisa.update_layout(height=500, width=1100, barmode='group')
fig_dist_lisa.write_json(OUTPUT_DIR / 'plot_lisa_distribution.json')
fig_dist_lisa.show()


## Save Output Files

In [43]:
df_global.to_csv(OUTPUT_DIR / 'output_global_morans_i.csv', index=False)
print('Saved: output_global_morans_i.csv')
print(df_global.to_string(index=False))

print()

df_lisa_out = df_lisa[[
    'year', 'province', 'poverty_rate',
    'local_moran_i', 'p_value', 'quadrant', 'significant', 'lon', 'lat'
]].sort_values(['year', 'province'])

df_lisa_out.to_csv(OUTPUT_DIR / 'output_lisa_results.csv', index=False)
print('Saved: output_lisa_results.csv')
print(df_lisa_out.head(20).to_string(index=False))

print(f'\nTotal rows output LISA: {len(df_lisa_out)}')
print(f'Columns: {df_lisa_out.columns.tolist()}')


Saved: output_global_morans_i.csv
 year  n_province  moran_i  expected_i  z_score      p_value  p_sim  significant
 2021          36 0.632166   -0.028571 7.194418 6.272783e-13  0.001         True
 2022          37 0.633053   -0.027778 7.305645 2.759396e-13  0.001         True
 2023          37 0.630284   -0.027778 7.275024 3.463594e-13  0.001         True
 2024          37 0.568522   -0.027778 6.592230 4.332676e-11  0.001         True

Saved: output_lisa_results.csv
 year            province  poverty_rate  local_moran_i  p_value quadrant  significant     lon    lat
 2021                BALI         4.165       0.262532    0.282       NS        False 115.188 -8.409
 2021              BANTEN         6.180       0.291004    0.193       NS        False 106.064 -6.406
 2021            BENGKULU        13.870      -0.159260    0.147       NS        False 102.346 -3.800
 2021       DI YOGYAKARTA        11.600       0.000522    0.097       NS        False 110.365 -7.800
 2021         DKI JAKART

In [44]:
print('SPATIAL AUTOCORRELATION ANALYSIS SUMMARY')
print()
print('GLOBAL MORAN\'S I:')
for _, row in df_global.iterrows():
    sig = 'SIGNIFICANT' if row['significant'] else 'not significant'
    print(f"  {int(row['year'])}: I = {row['moran_i']:.4f}, "
          f"z = {row['z_score']:.4f}, p = {row['p_sim']:.4f} [{sig}]")

print()
print('SIGNIFICANT LISA CLUSTERS (p<0.05):')
for year in sorted(df_lisa['year'].unique()):
    sub = df_lisa[(df_lisa['year'] == year) & (df_lisa['significant'])]
    hh = sub[sub['quadrant'] == 'HH']['province'].tolist()
    ll = sub[sub['quadrant'] == 'LL']['province'].tolist()
    hl = sub[sub['quadrant'] == 'HL']['province'].tolist()
    lh = sub[sub['quadrant'] == 'LH']['province'].tolist()
    print(f'  {year}:')
    if hh: print(f'    HH (hotspot): {hh}')
    if ll: print(f'    LL (coldspot): {ll}')
    if hl: print(f'    HL (high outlier): {hl}')
    if lh: print(f'    LH (low outlier): {lh}')
    if not (hh or ll or hl or lh): print('    No significant clusters')

print()
print('CSV Outputs:')
print('  1. output_global_morans_i.csv  -> Global Moran\'s I by year')
print('  2. output_lisa_results.csv     -> LISA by province by year')


SPATIAL AUTOCORRELATION ANALYSIS SUMMARY

GLOBAL MORAN'S I:
  2021: I = 0.6322, z = 7.1944, p = 0.0010 [SIGNIFICANT]
  2022: I = 0.6331, z = 7.3056, p = 0.0010 [SIGNIFICANT]
  2023: I = 0.6303, z = 7.2750, p = 0.0010 [SIGNIFICANT]
  2024: I = 0.5685, z = 6.5922, p = 0.0010 [SIGNIFICANT]

SIGNIFICANT LISA CLUSTERS (p<0.05):
  2021:
    HH (hotspot): ['PAPUA BARAT', 'PAPUA', 'PAPUA SELATAN', 'PAPUA TENGAH', 'PAPUA PEGUNUNGAN']
    LL (coldspot): ['KALIMANTAN TENGAH']
  2022:
    HH (hotspot): ['PAPUA BARAT', 'PAPUA', 'PAPUA SELATAN', 'PAPUA TENGAH', 'PAPUA PEGUNUNGAN']
    LL (coldspot): ['KALIMANTAN BARAT', 'KALIMANTAN TENGAH']
    HL (high outlier): ['ACEH']
  2023:
    HH (hotspot): ['PAPUA BARAT', 'PAPUA', 'PAPUA SELATAN', 'PAPUA TENGAH', 'PAPUA PEGUNUNGAN']
    LL (coldspot): ['KALIMANTAN BARAT', 'KALIMANTAN TENGAH']
    HL (high outlier): ['ACEH']
  2024:
    HH (hotspot): ['PAPUA BARAT', 'PAPUA BARAT DAYA', 'PAPUA', 'PAPUA SELATAN', 'PAPUA TENGAH', 'PAPUA PEGUNUNGAN']
    LL (cold